In [ ]:
!nvidia-smi

Wed Mar  4 05:53:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   43C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Need to install this version of protobuf first before installing anything else
!pip install "protobuf==5.29.5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 27.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.5
    Uninstalling protobuf-6.33.5:
      Successfully uninstalled protobuf-6.33.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-reflection 1.78.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 5.29.5 which is incompatible.
vllm 0.16.0 requires protobuf!=6.30.*,!=6.31.*,!=6.32.*,!=6.33.0.*,!=6.33.1.*,!=6.33.2.*,!=6.33.3.*,!=6.33.4.*,>=5.29.6, but you have protobuf 5.29.5 which is incompatible.


In [ ]:
# Install vLLM + utilities
!pip install -q vllm lm-format-enforcer pandas datasets scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.5 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.5 which is incompatible.


In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
import re
import json
import time
import numpy as np
from datasets import load_dataset
from tqdm import tqdm

print("Loading SuperGLUE BoolQ...")

# Use validation split (has labels); test split often has no labels in HF
ds = load_dataset("super_glue", "boolq", split="validation")

print(f"Dataset size: {len(ds)} examples")
print(f"Columns: {ds.column_names}")
# answer: True/False
print(f"Label distribution: {dict(zip(*np.unique([str(x) for x in ds['label']], return_counts=True)))}")

Loading SuperGLUE BoolQ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset size: 3270 examples
Columns: ['question', 'passage', 'idx', 'label']
Label distribution: {np.str_('0'): np.int64(1237), np.str_('1'): np.int64(2033)}


In [ ]:
# --- Config ---
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
MAX_NEW_TOKENS = 64
EVAL_SIZE = len(ds)     # set to e.g. 50 for a quick smoke test

SYSTEM_PROMPT = (
    "You are a yes/no question answering system. "
    "Given a passage and a question, answer with exactly one word: Yes or No."
)

print({
    "model": MODEL_NAME,
    "eval_size": EVAL_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
})

{'model': 'meta-llama/Meta-Llama-3-8B-Instruct', 'eval_size': 3270, 'max_new_tokens': 64}


In [ ]:
if __name__ == '__main__':
  from transformers import AutoTokenizer
  from vllm import LLM, SamplingParams

  print("Loading tokenizer...")
  tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

  print("Loading Llama3-8B with vLLM...")

  llm = LLM(
      model=MODEL_NAME,
      dtype="float16",
      gpu_memory_utilization=0.95,
      max_model_len=4096,
      enforce_eager=False,
  )

  sampling_params = SamplingParams(
      temperature=0,
      top_k=20,
      max_tokens=MAX_NEW_TOKENS,
  )

  print("Model loaded successfully!")

Loading tokenizer...
Loading Llama3-8B with vLLM...
INFO 03-04 05:55:31 [utils.py:223] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'model': 'meta-llama/Meta-Llama-3-8B-Instruct'}
INFO 03-04 05:55:31 [model.py:529] Resolved architecture: LlamaForCausalLM
WARNING 03-04 05:55:31 [model.py:1874] Casting torch.bfloat16 to torch.float16.
INFO 03-04 05:55:31 [model.py:1549] Using max model len 4096
INFO 03-04 05:55:31 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-04 05:57:55 [llm.py:355] Supported tasks: ['generate']
Model loaded successfully!


In [ ]:
# --- Output directory (Colab Drive if available) ---
IN_COLAB = True
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/Colab_Data/BoolQ/Llama3_8B_BoolQ_Eval_vLLM"
else:
    DRIVE_SAVE_DIR = os.path.abspath("./llama3_8b_boolq_eval_vllm_outputs")

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Saving results to: {DRIVE_SAVE_DIR}")

Mounted at /content/drive
Saving results to: /content/drive/MyDrive/Colab_Data/BoolQ/Llama3_8B_BoolQ_Eval_vLLM


In [ ]:
# --- Checkpoint setup ---
CHECKPOINT_FILE = os.path.join(DRIVE_SAVE_DIR, "llama3_8b_boolq_vllm_checkpoint.jsonl")
OUTPUTS_CACHE   = os.path.join(DRIVE_SAVE_DIR, "llama3_8b_boolq_vllm_raw_outputs.json")

results = []
if os.path.exists(CHECKPOINT_FILE):
    print(f"Found checkpoint: {CHECKPOINT_FILE}")
    with open(CHECKPOINT_FILE) as f:
        for line in f:
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    print(f"Loaded {len(results)} previously evaluated examples.")
else:
    print("No checkpoint found — starting from scratch.")

passages  = ds["passage"]
questions = ds["question"]
# Note: SuperGLUE BoolQ uses 'label' (int: 0=False, 1=True)
labels    = ds["label"]

already_done = len(results)
remaining_passages  = passages[already_done:EVAL_SIZE]
remaining_questions = questions[already_done:EVAL_SIZE]
remaining_labels    = labels[already_done:EVAL_SIZE]

def build_prompt(passage: str, question: str) -> str:
    # Standardized template similar to common BoolQ ICL benchmarks
    user_content = f"Passage: {passage}\nQuestion: {question}?\nAnswer:"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

prompts = [
    build_prompt(p, q) for p, q in zip(remaining_passages, remaining_questions)
]

print(f"Built {len(prompts)} prompts.")
if prompts:
    print("\nExample prompt (first 500 chars):")
    print(prompts[0][:500])
else:
    print("All requested examples are already processed.")

No checkpoint found — starting from scratch.
Built 3270 prompts.

Example prompt (first 500 chars):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a yes/no question answering system. Given a passage and a question, answer with exactly one word: Yes or No.<|eot_id|><|start_header_id|>user<|end_header_id|>

Passage: Ethanol fuel -- All biomass goes through at least some of these steps: it needs to be grown, collected, dried, fermented, distilled, and burned. All of these steps require resources and an infrastructure. The total amount of energy input into the process compare


In [ ]:
# --- Generate with vLLM ---
if prompts:
    print(f"Generating responses for {len(prompts)} prompts...")

    gen_start = time.time()
    outputs = llm.generate(prompts, sampling_params)
    gen_time = time.time() - gen_start

    total_new_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput = total_new_tokens / gen_time if gen_time > 0 else None

    print("\nGeneration complete.")
    print(f"  Time:       {gen_time/60:.1f} min")
    print(f"  Tokens:     {total_new_tokens:,}")
    print(f"  Throughput: {throughput:.1f} tokens/sec" if throughput else "  Throughput: N/A")

    raw_cache = [
        {
            "idx": already_done + i,
            "passage": remaining_passages[i],
            "question": remaining_questions[i],
            "ground_truth": bool(remaining_labels[i]), # Map 1 to True, 0 to False
            "response": o.outputs[0].text,
            "n_tokens": len(o.outputs[0].token_ids),
        }
        for i, o in enumerate(outputs)
    ]

    with open(OUTPUTS_CACHE, "w") as f:
        json.dump(raw_cache, f)
    print(f"Raw outputs cached to: {OUTPUTS_CACHE}")
else:
    print("No prompts to generate — loading cached outputs (if present).")
    if not os.path.exists(OUTPUTS_CACHE):
        raw_cache = []
    else:
        with open(OUTPUTS_CACHE) as f:
            raw_cache = json.load(f)
    gen_time = None
    throughput = None
    total_new_tokens = sum(r.get("n_tokens", 0) for r in raw_cache)

print(f"Raw items available for scoring: {len(raw_cache)}")

Generating responses for 3270 prompts...


Adding requests:   0%|          | 0/3270 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3270 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


Generation complete.
  Time:       2.5 min
  Tokens:     6,540
  Throughput: 44.2 tokens/sec
Raw outputs cached to: /content/drive/MyDrive/Colab_Data/BoolQ/Llama3_8B_BoolQ_Eval_vLLM/llama3_8b_boolq_vllm_raw_outputs.json
Raw items available for scoring: 3270


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[-1].strip()
    return text

def extract_bool(text: str):
    """Map model output to True/False. Returns None if unparseable."""
    text = strip_thinking(text).strip().lower()
    if "yes" in text and "no" not in text[: text.find("yes") + 3]:
        return True
    if text.startswith("yes"):
        return True
    if "no" in text and "yes" not in text[: text.find("no") + 2]:
        return False
    if text.startswith("no"):
        return False
    if "true" in text:
        return True
    if "false" in text:
        return False
    return None

# --- Score new outputs and append to checkpoint ---
new_results = []
for item in raw_cache:
    idx = item["idx"]
    response = item["response"]
    pred = extract_bool(response)
    gt = item["ground_truth"]

    res = {
        "idx": idx,
        "ground_truth": gt,
        "prediction": pred,
        "raw_response": response.strip(),
        "is_correct": (pred == gt) if pred is not None else False,
        "is_unknown": (pred is None),
        "new_tokens": int(item.get("n_tokens", 0)),
    }
    new_results.append(res)

if new_results:
    with open(CHECKPOINT_FILE, "a") as f:
        for r in new_results:
            f.write(json.dumps(r) + "\n")
    results.extend(new_results)

print(f"Total scored results: {len(results)}")

# --- Final metrics ---
all_gt = [r["ground_truth"] for r in results]
all_pred = [r["prediction"] if r["prediction"] is not None else "unknown" for r in results]
# For sklearn we need a label; treat unknown as wrong for accuracy
pred_for_acc = [r["prediction"] for r in results if r["prediction"] is not None]
gt_for_acc = [r["ground_truth"] for r in results if r["prediction"] is not None]

correct_count = sum(1 for r in results if r["is_correct"])
unknown_count = sum(1 for r in results if r["is_unknown"])
all_new_tokens = sum(r["new_tokens"] for r in results)
accuracy = correct_count / len(results) if results else 0
accuracy_known = accuracy_score(gt_for_acc, pred_for_acc) if pred_for_acc else 0

final_metrics = {
    "method": f"0_shot_vllm",
    "model": MODEL_NAME,
    "dataset": "super_glue/boolq:validation",
    "eval_size": len(results),
    "accuracy": accuracy,
    "accuracy_known_only": accuracy_known if pred_for_acc else "N/A",
    "unknown_frac": unknown_count / len(results) if results else 0,
    "total_new_tokens": all_new_tokens,
    "gen_tokens_per_sec": throughput if throughput is not None else "N/A (loaded from cache)",
    "total_gen_time_min": (gen_time / 60) if gen_time is not None else "N/A (loaded from cache)",
}

print("\n=== Final Metrics ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

print("\n=== Classification Report (True/False) ===")
labels = [True, False]
pred_labels = [p for p in all_pred if p != "unknown"]
gt_labels = [all_gt[i] for i in range(len(all_gt)) if all_pred[i] != "unknown"]
if pred_labels:
    print(classification_report(gt_labels, pred_labels, labels=labels))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(gt_labels, pred_labels, labels=labels))

METRICS_FILE = os.path.join(DRIVE_SAVE_DIR, "final_metrics.json")
with open(METRICS_FILE, "w") as f:
    json.dump(final_metrics, f, indent=2)
print(f"\nMetrics saved to: {METRICS_FILE}")

Total scored results: 3270

=== Final Metrics ===
  method: 0_shot_vllm
  model: meta-llama/Meta-Llama-3-8B-Instruct
  dataset: super_glue/boolq:validation
  eval_size: 3270
  accuracy: 0.845565749235474
  accuracy_known_only: 0.845565749235474
  unknown_frac: 0.0
  total_new_tokens: 6540
  gen_tokens_per_sec: 44.240016993122154
  total_gen_time_min: 2.463832688331604

=== Classification Report (True/False) ===
              precision    recall  f1-score   support

        True       0.92      0.82      0.87      2033
       False       0.75      0.89      0.81      1237

    accuracy                           0.85      3270
   macro avg       0.84      0.85      0.84      3270
weighted avg       0.86      0.85      0.85      3270

Confusion matrix (rows=true, cols=pred):
[[1668  365]
 [ 140 1097]]

Metrics saved to: /content/drive/MyDrive/Colab_Data/BoolQ/Llama3_8B_BoolQ_Eval_vLLM/final_metrics.json
